In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# Load the data
df = pd.read_csv('synthetic_triage_data.csv')
print(df.head()) # This will show you the first 5 rows of your data

   temperature  blood_pressure_sys  blood_pressure_dia  heart_rate  \
0         37.2                 120                  80          72   
1         39.5                 145                  95         115   
2         41.0                  90                  50         140   
3         36.8                 118                  78          68   
4         38.0                 130                  85          90   

                          chief_complaint  triage_priority  
0                         routine checkup                2  
1        severe fever and sweating chills                1  
2  unresponsive cold clammy skin no pulse                0  
3                           mild headache                2  
4                    cough and mild fever                2  


In [3]:
# defining features(x) and target(y) tell the model what data to look at (X) and what answer it is trying to predict (y).
x = df[['temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate', 'chief_complaint']] #our inputs
y = df[['triage_priority']] #our target ie priority level
#split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [4]:
#Machine learning models can only understand numbers,not text like 
#"severe fever".We use a *ColumnTransformer* to handle the numeric vitals 
#normally, but it uses a TfidfVectorizer(How It WorksTerm Frequency (TF): Counts how often a word appears in a specific document.Inverse Document Frequency (IDF): Reduces the score of common words (like "the" or "is") and increases the score of rare words.Combines both metrics to highlight words that are unique and meaningful to individual texts.) to convert the chief_complaint text
#into a matrix of numbers based on word frequency

#1. Define how to handle the different types of data
preprocessor = ColumnTransformer(
    transformers=[
        ('num','passthrough',['temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate']),
        ('text',TfidfVectorizer(),'chief_complaint')
    ]
)
#2. Create the pipeline: First process the data then feed it to the Random Forest model
#A Scikit-Learn Pipeline is exactly the same concept. It guarantees that our data flows in a strict sequence:
#Step 1: Run the data through the Pre-processor (translate text to numbers).
#Step 2: Feed those numbers into the Classifier (the AI model).
#3. What is happening inside the Random Forest Classifier?
#A "Decision Tree" is basically a giant, automatically generated if/else block.
#If Temp > 39 AND heart_rate > 100 AND symptoms contain "fever" -> Priority 1.
#But a single decision tree is prone to making mistakes. A Random Forest is 
# exactly what it sounds like: it generates hundreds of different decision 
#trees, gives them all slightly different parts of the data to look at, and 
#then has them vote on the final answer. It is incredibly powerful and highly
#accurate for tabular database records.
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])
#3. Train the model
pipeline.fit(x_train, y_train)
print("Model trained successfully!")

Model trained successfully!


/home/charity/PROJECTS/Portfolio/Healthcare-system/ai-triage-service/venv/lib64/python3.14/site-packages/sklearn/base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [6]:
# Save the trained pipeline as a file
joblib.dump(pipeline, 'triage_model.pkl')
print("Model saved as triage_model.pkl")

Model saved as triage_model.pkl


In [8]:
from sklearn.metrics import accuracy_score, classification_report
#force model to predict the test data
y_pred = pipeline.predict(x_test)
#check the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall accuracy: {accuracy * 100}%")
#detailed report
print("\nDetailed Report Card:")
print(classification_report(y_test, y_pred))

Overall accuracy: 100.0%

Detailed Report Card:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00        12

    accuracy                           1.00        21
   macro avg       1.00      1.00      1.00        21
weighted avg       1.00      1.00      1.00        21



In [15]:
#actual kaggle dataset
import pandas as pd
import numpy as np

In [16]:
df = pd.read_csv('data.csv',encoding='latin1', sep=';')
print(df.head()) # This will show you the first 5 rows of your data

   Group  Sex  Age  Patients number per hour  Arrival mode  Injury  \
0      2    2   71                         3             3       2   
1      1    1   56                        12             3       2   
2      2    1   68                         8             2       2   
3      1    2   71                         8             1       1   
4      1    2   58                         4             3       1   

       Chief_complain  Mental  Pain NRS_pain  ...    BT Saturation KTAS_RN  \
0   right ocular pain       1     1        2  ...  36.6        100       2   
1  right forearm burn       1     1        2  ...  36.5        NaN       4   
2        arm pain, Lt       1     1        2  ...  36.6         98       4   
3     ascites tapping       1     1        3  ...  36.5        NaN       4   
4     distension, abd       1     1        3  ...  36.5        NaN       4   

                                Diagnosis in ED Disposition KTAS_expert  \
0                              Corn

In [17]:
df_clean = df.rename(columns={
    'Age': 'age',
    'NRS_pain': 'pain_level',
    'BT': 'temperature',
    'SBP': 'blood_pressure_sys',
    'DBP': 'blood_pressure_dia',
    'HR': 'heart_rate',
    'Chief_complain': 'chief_complaint'
})

In [18]:
# 3. Map priority
def map_ktas_to_priority(ktas_score):
    if ktas_score in [1, 2]:
        return 0  
    elif ktas_score == 3:
        return 1  
    else:
        return 2  

df_clean['triage_priority'] = df_clean['KTAS_expert'].apply(map_ktas_to_priority)

# Filter out the extra columns we don't need (Sex, etc.)
df_clean = df_clean[['age', 'pain_level', 'temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate', 'chief_complaint', 'triage_priority']]
df_clean['chief_complaint'] = df_clean['chief_complaint'].fillna("unknown").astype(str)

# Clean numeric columns this converts any weird strings like "??" or blank spaces into true NaN values
vitals_columns = ['age', 'pain_level', 'temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate']
for col in vitals_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print(df_clean.head())

   age  pain_level  temperature  blood_pressure_sys  blood_pressure_dia  \
0   71         2.0         36.6               160.0               100.0   
1   56         2.0         36.5               137.0                75.0   
2   68         2.0         36.6               130.0                80.0   
3   71         3.0         36.5               139.0                94.0   
4   58         3.0         36.5                91.0                67.0   

   heart_rate     chief_complaint  triage_priority  
0        84.0   right ocular pain                2  
1        60.0  right forearm burn                2  
2       102.0        arm pain, Lt                2  
3        88.0     ascites tapping                2  
4        93.0     distension, abd                2  


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib

X = df_clean[['age', 'pain_level','temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate', 'chief_complaint']]
y = df_clean['triage_priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
# Imputer to handle missing blood pressure/heart rate values
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, ['age', 'pain_level','temperature', 'blood_pressure_sys', 'blood_pressure_dia', 'heart_rate']),
        ('text', TfidfVectorizer(max_features=5000), 'chief_complaint')
    ])

# Train the Random Forest
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)) 
])

print("\nTraining on real KTAS data...")
pipeline.fit(X_train, y_train)
print("Training complete!")
# Save the model
joblib.dump(pipeline, 'triage_model.pkl')
print("Model saved as triage_model.pkl")

# --- STEP 3: GRADE THE AI ---
from sklearn.metrics import accuracy_score, classification_report
y_pred = pipeline.predict(X_test)
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))


Training on real KTAS data...
Training complete!
Model saved as triage_model.pkl

Overall Accuracy: 73.23%

              precision    recall  f1-score   support

           0       0.82      0.56      0.67        55
           1       0.63      0.74      0.68        91
           2       0.81      0.81      0.81       108

    accuracy                           0.73       254
   macro avg       0.75      0.70      0.72       254
weighted avg       0.74      0.73      0.73       254

